In [1]:
from pytao import Tao, util
import numpy as np

In [2]:
init='-init tao.init -lat hxr_2cells.lat.bmad'
tao=Tao()
tao.init(init);

In [36]:
%%tao
sho ele mar.match

-------------------------
Tao> sho ele mar.match
 Element #               10
 Element Name: MAR.MATCH
 Key: Marker
 S_start, S:      3.557000,      3.557000
 Ref_time:  1.186488E-08

 Attribute values [Only non-zero/non-default values shown]:
   53   P0C                         =  3.0000000E+09 eV            BETA            =  0.999999985
   54   E_TOT                       =  3.0000000E+09 eV            GAMMA           =  5.8708536E+03

        TRACKING_METHOD             =  Bmad_Standard             APERTURE_AT               =  Exit_End
        MAT6_CALC_METHOD            =  Bmad_Standard             APERTURE_TYPE             =  Rectangular
        SPIN_TRACKING_METHOD        =  Tracking                  OFFSET_MOVES_APERTURE     =  F
        PTC_INTEGRATION_TYPE        =  Matrix_Kick

Slave_status: Free

Lord_status:  Not_a_Lord

Twiss at end of element:
                          A              B            Cbar                        C_mat
  Beta (m)        12.74140445     3.380389

In [30]:
def get_twiss(ele='Mar.MATCH'):
    res = tao.cmd('python ele:twiss '+ele+'|model')
    return util.parse_tao_python_data(res)
get_twiss()['alpha_a']

0.00079004728712051

In [31]:
def set_etot(e_tot):
    tao.cmd('change ele beginning e_tot @ '+str(e_tot))
set_etot(1.3789317E+10)    

In [32]:
def set_K(K_und, l_period=0.026):
    mc2 = 0.510998950e6
    c = 299792458.
    b_max = K_und * 2*np.pi*mc2 / (c*l_period)
    #print(b_max)
    cmd = 'change ele wiggler::* B_MAX @ '+str(b_max)
    tao.cmd(cmd)
    #print(cmd)
set_K(2)

In [37]:
def matching_optics(energy_GeV, K_und):
    set_etot(energy_GeV*1e9)
    set_K(K_und)
    t = get_twiss()
    return t
matching_optics(13.7,2)

{'mode_flip': False,
 'beta_a': 32.938239904269,
 'alpha_a': 0.00079004728712051,
 'gamma_a': 0.030359868258932,
 'phi_a': 0.12155157029757,
 'eta_a': 0.0,
 'etap_a': 0.0,
 'beta_b': 26.894978100721,
 'alpha_b': -5.9418812224155e-05,
 'gamma_b': 0.037181662680134,
 'phi_b': 0.11802915589562,
 'eta_b': 0.0,
 'etap_b': 0.0,
 'eta_x': 0.0,
 'etap_x': 0.0,
 'eta_y': 0.0,
 'etap_y': 0.0}

In [45]:
e_tot = 3
K = 2
t = matching_optics(e_tot, K)
with open('set_dat.tao', 'w') as f:
    f.write('change ele beginning e_tot @ '+str(e_tot)+'e9\n')
    f.write('set dat umatch[1]|meas = '+str(t['beta_a'])+'\n')
    f.write('set dat umatch[2]|meas = '+str(t['alpha_a'])+'\n')
    f.write('set dat umatch[3]|meas = '+str(t['beta_b'])+'\n')
    f.write('set dat umatch[4]|meas = '+str(t['alpha_b'])+'\n')

In [ ]:
for e in np.linspace(5, 14,20):
    t =  matching_optics(e, 2)
    print(e,t['beta_a'], t['alpha_a'])

In [ ]:
for k in np.linspace(1, 2.44, 10):
    t = matching_optics(14, k)
    print("{:.2f}".format(k), "{:.3f}".format(t['beta_a']), "{:.4f}".format(t['alpha_a']) )

In [ ]:
for k in np.linspace(1, 50000, 10):
    print(matching_optics(3, k), k)

# 2D scan

In [ ]:
dat = []
for k in np.linspace(1, 2.44,20):
    for e in np.linspace(3, 14,20):
        t=matching_optics(e,k)
        dat.append((e, k, t['beta_a'], t['beta_b']))
        

In [ ]:
#dat2 = np.array([res=matching_optics(e,k);(e, k, res['beta_a'], res['alpha_a']) for k in np.linspace(1, 2.44,100) for e in np.linspace(3, 14,100)])

In [ ]:
np.savetxt('hxr_matching_twiss.dat', dat)

# HXR Tools


HXR Quads
QHX13:QHX46
Odd: focusing
Even: defocusing



In [ ]:
for i in range(1,18):
    j = 11+i*2
    print('    var('+str(i)+')%ele_name = "QHX'+str(j)+'"')
for i in range(1,17):
    j = 12+i*2
    print('    var('+str(i)+')%ele_name = "QHX'+str(j)+'"')  